In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [29]:
import pandas as pd

ehr_path = '/content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/ehr/medical_transcriptions.csv'
df = pd.read_csv(ehr_path)

print("Total Records:", len(df))
print("\n Columns:", df.columns.tolist())
print("\n Missing Values:\n", df.isnull().sum())
print("\n Sample Rows:\n", df.head(3))


Total Records: 4999

 Columns: ['Unnamed: 0', 'description', 'medical_specialty', 'sample_name', 'transcription', 'keywords']

 Missing Values:
 Unnamed: 0              0
description             0
medical_specialty       0
sample_name             0
transcription          33
keywords             1068
dtype: int64

 Sample Rows:
    Unnamed: 0                                        description  \
0           0   A 23-year-old white female presents with comp...   
1           1           Consult for laparoscopic gastric bypass.   
2           2           Consult for laparoscopic gastric bypass.   

       medical_specialty                                sample_name  \
0   Allergy / Immunology                         Allergic Rhinitis    
1             Bariatrics   Laparoscopic Gastric Bypass Consult - 2    
2             Bariatrics   Laparoscopic Gastric Bypass Consult - 1    

                                       transcription  \
0  SUBJECTIVE:,  This 23-year-old white female pr...   


In [2]:
import pandas as pd

file_path = '/content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/ehr/medical_transcriptions.csv'
df = pd.read_csv(file_path)
print(df.head())


   Unnamed: 0                                        description  \
0           0   A 23-year-old white female presents with comp...   
1           1           Consult for laparoscopic gastric bypass.   
2           2           Consult for laparoscopic gastric bypass.   
3           3                             2-D M-Mode. Doppler.     
4           4                                 2-D Echocardiogram   

             medical_specialty                                sample_name  \
0         Allergy / Immunology                         Allergic Rhinitis    
1                   Bariatrics   Laparoscopic Gastric Bypass Consult - 2    
2                   Bariatrics   Laparoscopic Gastric Bypass Consult - 1    
3   Cardiovascular / Pulmonary                    2-D Echocardiogram - 1    
4   Cardiovascular / Pulmonary                    2-D Echocardiogram - 2    

                                       transcription  \
0  SUBJECTIVE:,  This 23-year-old white female pr...   
1  PAST MEDICAL 

In [7]:
import os

xray_dir = "/content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/xray"
print("Folders in X-ray dataset:", os.listdir(xray_dir))
print("Train subfolders:", os.listdir(os.path.join(xray_dir, 'train')))
print("Test subfolders:", os.listdir(os.path.join(xray_dir, 'test')))


Folders in X-ray dataset: ['val (1)', 'test (1)', 'train (1)', 'test', 'val', 'train']
Train subfolders: ['PNEUMONIA', 'NORMAL']
Test subfolders: ['PNEUMONIA', 'NORMAL']


In [8]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(rescale=1./255)
train_generator = train_datagen.flow_from_directory(
    os.path.join(xray_dir, 'train'),
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

Found 5216 images belonging to 2 classes.


In [9]:
df.info()
df.head(3)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4999 entries, 0 to 4998
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Unnamed: 0         4999 non-null   int64 
 1   description        4999 non-null   object
 2   medical_specialty  4999 non-null   object
 3   sample_name        4999 non-null   object
 4   transcription      4966 non-null   object
 5   keywords           3931 non-null   object
dtypes: int64(1), object(5)
memory usage: 234.5+ KB


,Unnamed: 0,description,medical_specialty,sample_name,transcription,keywords
0,0,A 23-year-old white female presents with comp...,Allergy / Immunology,Allergic Rhinitis,"SUBJECTIVE:, This 23-year-old white female pr...","allergy / immunology, allergic rhinitis, aller..."
1,1,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 2,"PAST MEDICAL HISTORY:, He has difficulty climb...","bariatrics, laparoscopic gastric bypass, weigh..."
2,2,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 1,"HISTORY OF PRESENT ILLNESS: , I have seen ABC ...","bariatrics, laparoscopic gastric bypass, heart..."


In [30]:
df = df.drop(columns=['Unnamed: 0'])

df = df.dropna(subset=['transcription'])

df = df.reset_index(drop=True)

print(f" Cleaned dataset shape: {df.shape}")
df.head(2)


 Cleaned dataset shape: (4966, 5)


,description,medical_specialty,sample_name,transcription,keywords
0,A 23-year-old white female presents with comp...,Allergy / Immunology,Allergic Rhinitis,"SUBJECTIVE:, This 23-year-old white female pr...","allergy / immunology, allergic rhinitis, aller..."
1,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 2,"PAST MEDICAL HISTORY:, He has difficulty climb...","bariatrics, laparoscopic gastric bypass, weigh..."


In [12]:
import os
from PIL import Image
import numpy as np
from tqdm import tqdm

SRC_BASE = "/content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/xray"
DST_BASE = "/content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/images_processed"
os.makedirs(DST_BASE, exist_ok=True)

TARGET = (256, 256)

# process each dataset split (train, test, val)
for split in ["train", "test", "val"]:
    src_split = os.path.join(SRC_BASE, split)
    dst_split = os.path.join(DST_BASE, split)
    os.makedirs(dst_split, exist_ok=True)

    print(f"\nProcessing {split} split...")
    for label in os.listdir(src_split):  # NORMAL / PNEUMONIA
        src_label = os.path.join(src_split, label)
        dst_label = os.path.join(dst_split, label)
        os.makedirs(dst_label, exist_ok=True)

        # iterate through all images
        for fn in tqdm(os.listdir(src_label), desc=f"{split}/{label}"):
            path = os.path.join(src_label, fn)
            try:
                img = Image.open(path).convert("L")  # grayscale
                img = img.resize(TARGET)
                outname = os.path.splitext(fn)[0] + ".png"
                img.save(os.path.join(dst_label, outname))
            except Exception as e:
                print("Skipping", fn, ":", e)

print("\nAll X-ray images processed and standardized!")



Processing train split...


train/NORMAL: 100%|██████████| 1341/1341 [04:37<00:00,  4.83it/s]



Processing test split...


test/NORMAL: 100%|██████████| 234/234 [00:14<00:00, 15.86it/s]



Processing val split...


val/PNEUMONIA: 100%|██████████| 8/8 [00:02<00:00,  3.63it/s]


All X-ray images processed and standardized!


In [15]:
import os
import pandas as pd

ehr_csv = "/content/drive/MyDrive/Enhancing_EHRs_with_GenAI/data/ehr/medical_transcriptions.csv"

ehr_text_dir = "/content/Enhancing_EHRs_with_GenAI/data/ehr_notes_processed"
os.makedirs(ehr_text_dir, exist_ok=True)

df = pd.read_csv(ehr_csv)

text_col = None
for col in ['transcription', 'description', 'medical_specialty', 'keywords']:
    if col in df.columns:
        text_col = col
        break

if text_col is None:
    raise ValueError("No text column found. Check your CSV columns!")

for i, text in enumerate(df[text_col].astype(str)):
    cleaned = text.replace('\r', '\n').strip()
    out_path = os.path.join(ehr_text_dir, f"note_{i+1:04d}.txt")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(cleaned)

print(f"Saved {len(df)} cleaned EHR text files to {ehr_text_dir}")


Saved 4999 cleaned EHR text files to /content/Enhancing_EHRs_with_GenAI/data/ehr_notes_processed


In [16]:
import glob
import pandas as pd
import os


images = sorted(glob.glob("/content/Enhancing_EHRs_with_GenAI/data/images_processed/train/NORMAL/*.png") +
                glob.glob("/content/Enhancing_EHRs_with_GenAI/data/images_processed/train/PNEUMONIA/*.png"))
notes = sorted(glob.glob("/content/Enhancing_EHRs_with_GenAI/data/ehr_notes_processed/*.txt"))

n = min(len(images), len(notes))
images = images[:n]
notes = notes[:n]

rows = []
for i in range(n):
    pid = f"{i+1:04d}"
    rows.append({
        "file_id": pid,
        "image_path": images[i],
        "note_path": notes[i],
        "diagnosis": "",
        "icd10": ""
    })


out_path = "/content/Enhancing_EHRs_with_GenAI/data/mapping.csv"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
pd.DataFrame(rows).to_csv(out_path, index=False)

print(f" Created mapping.csv with {n} records")
print(f" Saved to: {out_path}")


 Created mapping.csv with 2612 records
 Saved to: /content/Enhancing_EHRs_with_GenAI/data/mapping.csv


In [17]:
import glob

processed_images = glob.glob("/content/Enhancing_EHRs_with_GenAI/data/images_processed/**/*.png", recursive=True)
print("Total processed images:", len(processed_images))

print("\nExample files:")
print(processed_images[:5])


Total processed images: 2612

Example files:
['/content/Enhancing_EHRs_with_GenAI/data/images_processed/train/PNEUMONIA/person478_bacteria_2035.png', '/content/Enhancing_EHRs_with_GenAI/data/images_processed/train/PNEUMONIA/person946_bacteria_2871.png', '/content/Enhancing_EHRs_with_GenAI/data/images_processed/train/PNEUMONIA/person1513_bacteria_3962.png', '/content/Enhancing_EHRs_with_GenAI/data/images_processed/train/PNEUMONIA/person1493_bacteria_3896.png', '/content/Enhancing_EHRs_with_GenAI/data/images_processed/train/PNEUMONIA/person731_bacteria_2633.png']


In [22]:
import pandas as pd
import os

lookup_data = {
    "condition_keyword": ["pneumonia", "hypertension", "diabetes", "asthma", "fracture"],
    "icd10_code": ["J18.9", "I10", "E11.9", "J45.909", "S52.501A"],
    "icd10_description": [
        "Pneumonia, unspecified organism",
        "Essential (primary) hypertension",
        "Type 2 diabetes mellitus without complications",
        "Unspecified asthma, uncomplicated",
        "Unspecified fracture of the lower end of right radius"
    ]
}

df_lookup = pd.DataFrame(lookup_data)

os.makedirs("/content/Enhancing_EHRs_with_GenAI/data", exist_ok=True)
csv_path = "/content/Enhancing_EHRs_with_GenAI/data/icd_lookup.csv"
df_lookup.to_csv(csv_path, index=False)

print(f"Created ICD-10 lookup file at: {csv_path}")
print(df_lookup)


Created ICD-10 lookup file at: /content/Enhancing_EHRs_with_GenAI/data/icd_lookup.csv
  condition_keyword icd10_code  \
0         pneumonia      J18.9   
1      hypertension        I10   
2          diabetes      E11.9   
3            asthma    J45.909   
4          fracture   S52.501A   

                                   icd10_description  
0                    Pneumonia, unspecified organism  
1                   Essential (primary) hypertension  
2     Type 2 diabetes mellitus without complications  
3                  Unspecified asthma, uncomplicated  
4  Unspecified fracture of the lower end of right...  


In [25]:
import pandas as pd


mapping = pd.read_csv("/content/Enhancing_EHRs_with_GenAI/data/mapping.csv")
lookup = pd.read_csv("/content/Enhancing_EHRs_with_GenAI/data/icd_lookup.csv")

def suggest_icd(note_text):
    t = str(note_text).lower()
    for _, r in lookup.iterrows():
        if r["condition_keyword"] in t:
            return r["icd10_code"]
    return "UNKNOWN"

icd_codes = []
diagnoses = []

for path in mapping["note_path"]:
    try:
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
        code = suggest_icd(text)
        icd_codes.append(code)

        diagnosis = next(
            (r["condition_keyword"] for _, r in lookup.iterrows() if r["condition_keyword"] in text.lower()),
            "unspecified",
        )
        diagnoses.append(diagnosis)
    except:
        icd_codes.append("UNKNOWN")
        diagnoses.append("unspecified")

mapping["diagnosis"] = diagnoses
mapping["icd10"] = icd_codes


mapping.to_csv("/content/Enhancing_EHRs_with_GenAI/data/mapping.csv", index=False)

print(" Updated mapping.csv with ICD-10 codes and diagnoses")
mapping.head()


 Updated mapping.csv with ICD-10 codes and diagnoses


,file_id,image_path,note_path,diagnosis,icd10
0,1,/content/Enhancing_EHRs_with_GenAI/data/images...,/content/Enhancing_EHRs_with_GenAI/data/ehr_no...,asthma,J45.909
1,2,/content/Enhancing_EHRs_with_GenAI/data/images...,/content/Enhancing_EHRs_with_GenAI/data/ehr_no...,hypertension,I10
2,3,/content/Enhancing_EHRs_with_GenAI/data/images...,/content/Enhancing_EHRs_with_GenAI/data/ehr_no...,hypertension,I10
3,4,/content/Enhancing_EHRs_with_GenAI/data/images...,/content/Enhancing_EHRs_with_GenAI/data/ehr_no...,unspecified,UNKNOWN
4,5,/content/Enhancing_EHRs_with_GenAI/data/images...,/content/Enhancing_EHRs_with_GenAI/data/ehr_no...,hypertension,I10


In [26]:
import pandas as pd
m = pd.read_csv("/content/Enhancing_EHRs_with_GenAI/data/mapping.csv")
print("Total records:", len(m))
print("Unique ICD-10 codes:", m['icd10'].nunique())
m['icd10'].value_counts().head()


Total records: 2612
Unique ICD-10 codes: 6


,count
icd10,
UNKNOWN,2139
S52.501A,205
I10,136
E11.9,61
J18.9,45


In [27]:
import pandas as pd, os, re

m = pd.read_csv('/content/Enhancing_EHRs_with_GenAI/data/mapping.csv')

missing_images = m[~m['image_path'].apply(os.path.exists)]
missing_notes  = m[~m['note_path'].apply(os.path.exists)]

print("Missing images:", len(missing_images))
print("Missing notes:", len(missing_notes))

pattern = re.compile(r'^[A-Z][0-9]{2}(?:\.[0-9A-Za-z]{1,4})?$')
bad_icd = m[~m['icd10'].apply(lambda x: bool(pattern.match(str(x))) or x == '' or x == 'UNKNOWN')]

print("Bad ICD codes:", len(bad_icd))

print("\n Sample of valid entries:")
print(m.sample(5))


Missing images: 0
Missing notes: 0
Bad ICD codes: 0

 Sample of valid entries:
      file_id                                         image_path  \
190       191  /content/Enhancing_EHRs_with_GenAI/data/images...   
1546     1547  /content/Enhancing_EHRs_with_GenAI/data/images...   
2425     2426  /content/Enhancing_EHRs_with_GenAI/data/images...   
1561     1562  /content/Enhancing_EHRs_with_GenAI/data/images...   
635       636  /content/Enhancing_EHRs_with_GenAI/data/images...   

                                              note_path    diagnosis     icd10  
190   /content/Enhancing_EHRs_with_GenAI/data/ehr_no...     fracture  S52.501A  
1546  /content/Enhancing_EHRs_with_GenAI/data/ehr_no...  unspecified   UNKNOWN  
2425  /content/Enhancing_EHRs_with_GenAI/data/ehr_no...  unspecified   UNKNOWN  
1561  /content/Enhancing_EHRs_with_GenAI/data/ehr_no...     fracture  S52.501A  
635   /content/Enhancing_EHRs_with_GenAI/data/ehr_no...  unspecified   UNKNOWN  
